In [1]:
# B-1 master table  ->  data/meta/master.csv
import os, json, shutil
from pathlib import Path
import numpy as np, pandas as pd

root = Path.cwd()
while not (root / "data" / "imgs_cleaned").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
CLEAN = Path("data/imgs_cleaned"); A1 = Path("logs/preprocess_a1"); META = Path("data/meta")

# logs/ is git-ignored, so keep the two files the build depends on under data/meta
for n in ("letterbox.csv", "black_boxes.csv"): shutil.copy(A1 / n, META / n)

a1 = pd.read_csv(A1 / "measurements.csv")[["file", "cls", "source", "w", "h", "bytes", "sha1",
                                           "bright", "contrast", "blur", "chroma", "flags"]]
near = pd.read_csv(META / "near_dup_groups.csv")[["file", "label", "group_id", "nn_sim", "nn_file"]]
rv = pd.read_csv(META / "review.csv")
drop = set(rv.loc[rv.action == "drop", "file"])

t = near.merge(a1, on="file", how="left", validate="1:1")
assert t["cls"].notna().all() and not (set(t["file"]) & drop), "master/near-dup/review lists disagree"
assert set(t["label"]) == set(json.load(open(META / "classes.json")).values()), "class names differ from classes.json"

# photo sessions: dedup groups that belong to the same home photo session get merged
SESSION_MERGES = [("rice/rice_0128.jpg", "rice/rice_0157.jpg")]      # red tray + red bowl, same cloth and day
for a, b in SESSION_MERGES:
    ga = t.loc[t["file"] == a, "group_id"].iat[0]; gb = t.loc[t["file"] == b, "group_id"].iat[0]
    t.loc[t["group_id"] == gb, "group_id"] = ga
t["group_size"] = t.groupby("group_id")["file"].transform("size")
t["session"] = np.where(t["group_size"] >= 15, t["group_id"], "")      # big groups = photo sessions

box = set(pd.read_csv(META / "black_boxes.csv")["file"])
lb = pd.read_csv(META / "letterbox.csv").set_index("file")[["x0", "y0", "x1", "y1"]]
t["has_box"] = t["file"].isin(box)
t = t.merge(lb, left_on="file", right_index=True, how="left")
t["letterbox"] = t["x0"].notna()
t["note"] = t["file"].map(rv[rv.action == "note"].set_index("file")["detail"]).fillna("")
num = t["file"].str.extract(r"_(\d+)\.jpg$")[0].astype(int)
t["style"] = np.where((t["label"] == "pittu") & (num >= 33), "processed", "")   # HDR-looking pittu (visual review + blur score)
t["folder"] = t["file"].str.split("/").str[0]

cols = ["file", "label", "folder", "source", "group_id", "group_size", "session", "w", "h", "has_box", "letterbox",
        "x0", "y0", "x1", "y1", "style", "note", "flags", "nn_sim", "nn_file", "bytes", "sha1",
        "bright", "contrast", "blur", "chroma"]
t = t[cols].sort_values("file").reset_index(drop=True)
t.to_csv(META / "master.csv", index=False)

print(len(t), "images ->", META / "master.csv\n")
print(t.groupby("label").agg(images=("file", "size"), groups=("group_id", "nunique"), biggest=("group_size", "max"),
                             sessions=("session", lambda s: s[s != ""].nunique()),
                             boxes=("has_box", "sum"), crops=("letterbox", "sum")).to_string())
print("\nrelabelled:", t.loc[t["label"] != t["folder"], ["file", "label"]].values.tolist())
print("pittu with processed style:", int((t["style"] == "processed").sum()), "of", int((t["label"] == "pittu").sum()))
print("sessions:", t[t["session"] != ""].groupby("session").agg(n=("file", "size"), first=("file", "min"), last=("file", "max")).to_string())

1357 images -> data\meta\master.csv

                images  groups  biggest  sessions  boxes  crops
label                                                          
dosa               176     164        3         0      0      2
hoppers            156     118        3         0      2     32
idli                85      84        2         0      0      2
kottu               66      59        3         0      1      0
milk_rice          143     108        3         0      2     31
paratha            114     101        3         0      0      4
pittu              246     209        5         0      0      3
rice               288     161       49         3     26      0
string_hoppers      83      78        3         0      0      1

relabelled: [['dosa/dosa_0105.jpg', 'idli']]
pittu with processed style: 214 of 246
sessions:           n               first                last
session                                            
g0852    16  rice/rice_0011.jpg  rice/rice_0026.jpg
g0853   